In [ ]:
import pandas as pd
import numpy as np
from imagecodecs import NoneError
from scipy.stats import hmean
import matplotlib.pyplot as plt
import os
from scipy import stats

In [ ]:
plt.style.use("default")

In [ ]:
"""Variables influençant le comportement du notebook"""
scale_system = 'pWWP_HS' # 'ADeLe' pour travailler directement avec les niveaux ADeLe, 'pWWP_HS' pour travailler dans l'échelle du papier 'Human Scales' avec les coefficients issus du preprint
data_display = 'verbose' #Concerne le texte des scatterplots. 'threshold only' pour voir seulement une date, '' pour les données de régression avec estimation d'incertitude

In [ ]:
'''Lire les ability profiles, regrouper ceux du nature et les plus récents'''
nature_profiles_df = pd.read_csv("ability_profiles/ability_profiles_MaxAuxDiff=0_DupThresh=100.csv", index_col=0)
additional_profiles_df = pd.read_csv('data/additional_results_16June2026.csv', index_col=0)
ability_profiles_df = pd.concat([nature_profiles_df, additional_profiles_df], axis=0)

In [ ]:
'''Collapser les ability profiles sur les groupes de dimensions du human scales'''
#On exploite la nomenclature des auteurs, dans laquelle chaque nom de groupe est constitué de deux lettres majuscules qui sont les deux premières lettres des dimensions qui le constituent
#Une exception à traiter à part : SNs
dimension_groups = list(set([name[:2] for name in ability_profiles_df.columns]))
dimension_groups.sort()
dimensions = list(ability_profiles_df.columns)
#groupping_dict = {name : name[:2] for name in dimensions}
for dimension_group in dimension_groups:
    components = [dimension for dimension in dimensions if dimension[:2] == dimension_group]
    if len(components) >= 2:
        ability_profiles_df[dimension_group] = ability_profiles_df[components].apply(hmean, axis=1)
ability_profiles_df['SN'] = ability_profiles_df['SNs'].copy()
reduced_profiles_columns = list(dimension_groups)
reduced_profiles_df = ability_profiles_df[reduced_profiles_columns].copy()

In [ ]:
if scale_system == 'pWWP_HS':
    '''Changement d'échelle avec les coefficients fournis par le human scales'''
    rescaling_coefficients_df = pd.read_csv("data/rescaling_coefficients.csv")
    rescaled_profiles_df = reduced_profiles_df.copy()
    dimension_groups = [dimension_group for dimension_group in dimension_groups if
                    dimension_group in rescaling_coefficients_df.columns]
    for dimension_group in dimension_groups:
        rescaled_profiles_df[dimension_group] = rescaled_profiles_df[dimension_group].apply(
            lambda x: rescaling_coefficients_df[dimension_group].iloc[0] * x +
                rescaling_coefficients_df[dimension_group].iloc[1])
elif scale_system == 'ADeLe':
    rescaled_profiles_df = reduced_profiles_df.copy() # Attention : pour des raisons pratiques, on travaille avec un dataframe nommé rescaled_profiles_df, qu'on aie fait d'une manière ou d'une autre ou ignoré l'etape de changement d'échelle. Possiblement sujet à modifications ultérieures.
else:
    raise Exception(f'Unknown scale system : {scale_system}')

In [ ]:
release_dates_df = pd.read_csv("data/release_dates.csv", index_col=0)
release_dates_df = release_dates_df[['Date de sortie']]
release_dates_df['Date de sortie'] = pd.to_datetime(release_dates_df['Date de sortie'])
rescaled_profiles_df['Date'] = release_dates_df['Date de sortie']

In [ ]:
def frontier_indices(xs, ys): #Vérifier le traitement des cas d'égalité. Il faudrait idéalement contrôler si plusieurs modèles sortis le même jour et dépassant ceux qui les précèdent sont tous frontière, ou seulement le meilleur d'entre eux. La version actuelle du code ne me semble pas donner de garantie à ce sujet.
    assert len(xs) == len(ys)
    sorted_order = np.argsort(xs)
    current_max = -np.inf
    front_original_indices = []
    for idx in sorted_order:
        if ys[idx] > current_max:
            current_max = ys[idx]
            front_original_indices.append(idx)
    return front_original_indices

In [ ]:
corporations = {'Babbage-002' : 'OpenAI',
                'Davinci-002' : 'OpenAI',
                'GPT-3.5-Turbo' : 'OpenAI',
                'GPT-4o' : 'OpenAI',
                'OpenAI o1-mini' : 'OpenAI',
                'OpenAI o1' : 'OpenAI',
                'LLaMA-3.2-1B-Instruct' : 'Meta',
                'LLaMA-3.2-3B-Instruct' : 'Meta',
                'LLaMA-3.2-11B-Instruct' : 'Meta',
                'LLaMA-3.2-90B-Instruct' : 'Meta',
                'LLaMA-3.1-405B-Instruct' : 'Meta',
                'DK-R1-Dist-Qwen-1.5B' : 'Qwen',
                'DK-R1-Dist-Qwen-7B' : 'Qwen',
                'DK-R1-Dist-Qwen-14B' : 'Qwen',
                'DK-R1-Dist-Qwen-32B' : 'Qwen',
                'Gemini-2.5-Flash' : 'Google',
                'Gemini-3.1-Flash' : 'Google',
                'Gemini-3.1-Pro' : 'Google',
                'LLaMA-4-17B-128E' : 'Meta',
                'GPT-5.2-Chat' : 'OpenAI',
                'OpenAI o3-mini' : 'OpenAI'}


In [ ]:
corpo_color_map = {'OpenAI' : 'r',
                   'Meta' : 'b',
                   'Qwen' : 'g',
                   'Google' : 'purple'}

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(16, 15), sharex=True, sharey=True)
demand_groups = list(rescaled_profiles_df.columns)
demand_groups.remove('Date')
release_dates = rescaled_profiles_df['Date'].values
model_names = list(rescaled_profiles_df.index)
xs = release_dates
colors = [corpo_color_map[corporations[x]]  for x in model_names]

for i in range(3):
    for j in range(3):
        ax = axs[i][j]
        ys = rescaled_profiles_df[demand_groups[3*i+j]].values
        front = set(frontier_indices(xs, ys))
        alphas, xsfront, ysfront = [], [], []
        for k in range(len(xs)):
            if k in front:
                alphas.append(1)
                xsfront.append(xs[k])
                ysfront.append(ys[k])
            else : alphas.append(0.5)
        xsfront_years_since_2022 = [(x - pd.to_datetime("2022-01-01")).total_seconds()/(3600*24*365.25) for x in xsfront]

        ax.scatter(xs, ys, zorder=3, c=colors, alpha=alphas)

        # Labels frontière
        for k in front:
            ax.annotate(
                model_names[k], (xs[k], ys[k]),
                textcoords="offset points", xytext=(0, 6),
                ha='center', fontsize=7
            )

        regression = stats.linregress(xsfront_years_since_2022, ysfront, alternative='greater')
        slope, intercept, r, p_value, slope_stderr = regression
        intercept_stderr = regression.intercept_stderr
        r2 = r*r
        #slope, intercept = np.polyfit(xsfront_years_since_2022, ysfront, deg=1)
        x_0, x_1 = (min(xsfront) - pd.to_datetime("2022-01-01")).total_seconds()/(3600*24*365.25), (max(xsfront) - pd.to_datetime("2022-01-01")).total_seconds()/(3600*24*365.25)
        y_0, y_1 = intercept + slope * x_0, intercept + slope * x_1
        ax.plot([min(xsfront), max(xsfront)], [y_0, y_1], ls='dashdot', color='black')

        # Attention : les cas où la pente est nulle ou strictement négative ne sont pas gérés ici, et donneraient respectivement une erreur ou un résultat qu'il faut interpréter différemment du cas où la pente est positive. De même, le cas où la pente est inférieure à son incertitude-type pose problème.

        x_millionth = (6-intercept)/slope
        date_millionth = pd.to_datetime("2022-01-01") + pd.to_timedelta(x_millionth*365.25, unit='D')
        x_millionth_under = (6-intercept)/(slope+slope_stderr)
        x_millionth_above = (6-intercept)/(slope-slope_stderr)
        delta_x_millionth_left = x_millionth - x_millionth_under
        delta_x_millionth_right = x_millionth_above - x_millionth
        date_millionth_under = pd.to_datetime("2022-01-01") + pd.to_timedelta(x_millionth_under*365.25, unit='D')
        date_millionth_above = pd.to_datetime("2022-01-01") + pd.to_timedelta(x_millionth_above*365.25, unit='D')

        mois_fr = {
                    1: 'Janvier', 2: 'Février', 3: 'Mars', 4: 'Avril', 5: 'Mai', 6: 'Juin',
                    7: 'Juillet', 8: 'Août', 9: 'Septembre', 10: 'Octobre', 11: 'Novembre', 12: 'Décembre'
        }

        if data_display == 'verbose':
            stats_text = (
            f"slope: {slope:.2f} B/yr ± {slope_stderr:.2f} \n"
            f"Jan 2022: {intercept:.2f} ± {intercept_stderr:.2f}\n"
            f"r2: {r2:.2f}\n"
            f"p: {p_value:.1e}\n"
            f"lvl 6 : {date_millionth.month}-{date_millionth.year}\n"
            f"(± {intercept_stderr*12/slope: .1f} ; - {delta_x_millionth_left*12: .1f} / + {delta_x_millionth_right*12 : .1f}) m"
            )
            ax.text(0.05, 0.95, stats_text, transform=ax.transAxes,
            fontsize=13,
            verticalalignment='top', horizontalalignment='left',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.5),
            color='black')

        elif data_display == 'threshold_only':

            stats_text = (
            f"Niveau 6 attendu en {mois_fr[date_millionth.month]} {date_millionth.year}"
            )

            ax.text(0.05, 0.95, stats_text, transform=ax.transAxes,
            fontsize=13,
            verticalalignment='top', horizontalalignment='left',
            bbox=dict(boxstyle='round', facecolor='r', alpha=0.5),
            color='black')

        else:
            raise Exception(f"data_display '{data_display}' not recognized")

        ax.set_title(demand_groups[3*i+j])

        # Grille gris clair
        ax.grid(True, color='lightgrey', linewidth=0.7, zorder=0)
        ax.set_axisbelow(True)

        # Rotation xticks
        ax.tick_params(axis='x', rotation=45)

scale_system_displays = {'ADeLe' : 'Niveau ADeLe', 'pWWP_HS' : r'$-\text{log]}(p^{WWP})$ (selon coeffs preprint)'}
fig.supylabel(f'{scale_system_displays[scale_system]}', fontsize=18)
plt.tight_layout()
plt.show()